# NatureCubePy Data Upload Tutorial

This notebook demonstrates safe, practical upload workflows for NatureCubePy.

The examples default to dry-run mode so you can inspect payloads before sending any writes to the API.

In [2]:
from datetime import datetime, timedelta, timezone
import uuid

import pandas as pd

from naturecubepy import (
    add_project_labels,
    auth_headers,
    build_device_settings,
    build_feature_record,
    build_observation,
    check_edna_labels_df,
    get_camera_trap_data,
    get_key,
    get_project,
    get_project_labels,
    get_station_info,
    get_media_assets_df,
    set_segment_published_status,
    update_media_timestamps,
    upload_edna_records,
    upload_phone_observations,
    validate_observation_payload,
)

In [11]:
# Retrieve API key and set up authentication headers
api_key = get_key('EV2_SIT')
hdr = auth_headers(api_key, okala_url='http://127.0.0.1:8000/api')
project_name = get_project(hdr)

Retrieving project data...
Received response with status code 401


HTTPStatusError: Client error '401 Unauthorized' for url 'http://127.0.0.1:8000/api/getProject/fTioYojcV3lffOeWUUjtAzpjGp3q2EAcDh4OReYGx6tEPLpcogXWBewIP4WnrYTAYaSGJihcwzlL9ocqUm8bOzGI5TtqB2pWCRRM'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

In [ ]:
# Retrieve API key and set up authentication headers
api_key = get_key('EV2_PROD')
hdr = auth_headers(api_key)   

In [6]:
# Get station data and associated media data
stations = get_station_info(hdr, measurement_type='all')
psr_ids = stations["project_system_record_id"]

In [7]:
len(stations)

18

In [ ]:
media_assets = get_media_assets_df(hdr, "audio", psr_ids)
display(media_assets.head())

In [23]:
media_assets.device_id.unique()

<StringArray>
['0068W1ARU1', '0068W2ARU2', '0068W3ARU3', '0068W4ARU4', '0068W5ARU5',
 '0068W6ARU6',  '0068W1WC1',  '0068W2WC2',  '0068W3WC3',  '0068W4WC4',
  '0068W5WC5',  '0068W6WC6']
Length: 12, dtype: str

In [24]:
info = pd.read_excel('/Users/natimi/Downloads/Okala_birds_SFL_combined.xlsx')
us = info[info.Country == 'USA']

publish_map = us.set_index('Species')['Publish Detections']
species_lookup = media_assets.species.map(publish_map)

unpublish_mask = (
    (species_lookup == 'none') |
    ((species_lookup == 'all above 95%') & (media_assets['prediction_accuracy'] < 95)) |
    ((species_lookup == 'only human verified') & (media_assets.segment_verification_status == 'ai_derived'))
)

In [27]:
for status, mask in [(False, unpublish_mask), (True, ~unpublish_mask)]:
    set_segment_published_status(hdr, published_status=status, segment_record_ids=media_assets.loc[mask, 'segment_record_id'])

Publish status updated successfully
Publish status updated successfully


In [ ]:
# get segment id of observation to change
seg_id = media_assets[(media_assets.species == 'Odocoileus virginianus')]['segment_record_id']

In [7]:
r = set_segment_published_status(hdr, published_status=True, segment_record_ids=seg_id)

Publish status updated successfully


## 1. Upload Project Labels

This example creates `Label` payloads and uploads them with `add_project_labels(...)`.

To keep the example realistic and safe, we reuse a few existing camera labels from your project.

In [ ]:
camera_labels = get_project_labels(hdr, "Camera")
print(f"Loaded {len(camera_labels)} existing camera labels")

if camera_labels.empty:
    print("No project labels found. Add labels in the dashboard first, then rerun this cell.")


Loaded 5 existing camera labels
Prepared 3 label record(s)
Dry run only: set RUN_UPLOADS=True to call add_project_labels(...)


In [7]:
add_project_labels(hdr, "Camera", labels=camera_labels)

AttributeError: 'str' object has no attribute 'model_dump'

## 2. Upload Timestamp Corrections

Use `update_media_timestamps(...)` with a list of `MediaTimestampUpdate` objects.

This example shifts a small sample of media timestamps by +1 minute.

In [ ]:
camera_obs = get_camera_trap_data(hdr)
if camera_obs.empty:
    print("No camera observations found.")
else:
    sample = camera_obs[["media_file_record_id", "media_file_created_at"]].dropna().drop_duplicates().head(5)
    updates = []
    for row in sample.itertuples(index=False):
        old_ts = pd.to_datetime(row.media_file_created_at, utc=True)
        new_ts = (old_ts + timedelta(minutes=1)).to_pydatetime()
        updates.append(MediaTimestampUpdate(media_file_record_id=int(row.media_file_record_id), new_timestamp=new_ts))

    print(f"Prepared {len(updates)} timestamp update record(s)")

    if RUN_UPLOADS and updates:
        results = update_media_timestamps(hdr, updates)
        print(f"API returned {len(results)} responses")
    else:
        print("Dry run only: set RUN_UPLOADS=True to call update_media_timestamps(...)")

## 3. Validate and Upload eDNA Records

Recommended flow:
1. Build eDNA rows in a DataFrame
2. Validate with `check_edna_labels_df(...)`
3. Upload only rows with `status == 'success'`

In [ ]:
edna_input = pd.DataFrame([
    {
        "marker_name": "COI",
        "sequence": "ATGCCGTAGCTA",
        "primer": "mlCOIintF",
        "timestamp": datetime.now(tz=timezone.utc),
        "genus": "Canis",
        "species": "Canis lupus",
        "confidence": 99,
    },
])

validated_df = check_edna_labels_df(hdr, edna_input)
display(validated_df)

successful_records = [
    eDNAUploadResponse.model_validate(record)
    for record in validated_df.to_dict(orient="records")
    if record.get("status") == "success"
]
print(f"Validated success rows: {len(successful_records)}")

edna_stations = get_station_info(hdr, measurement_type="edna")
target_psr = int(edna_stations.project_system_record_id.dropna().iloc[0]) if not edna_stations.empty else None
print(f"Target eDNA project_system_record_id: {target_psr}")

if RUN_UPLOADS and successful_records and target_psr is not None:
    upload_result = upload_edna_records(hdr, successful_records, project_system_record_id=target_psr)
    print(f"Uploaded/processed rows: {len(upload_result)}")
else:
    print("Dry run only: set RUN_UPLOADS=True to call upload_edna_records(...)")

## 4. Upload Phone Observations

This example creates a single text observation, validates it, and optionally uploads it.

Set `PROJECT_ID` to your numeric project ID before upload.

In [ ]:
PROJECT_ID = None  # e.g. 42

device = build_device_settings(
    device_id=f"demo-{uuid.uuid4().hex[:8]}",
    phone_model="iPhone 14 Pro",
    phone_os="iOS 17",
    carrier="Demo Carrier",
    build_number="1.0.0",
    build_id="tutorial-build",
)

observation = build_observation(
    item_uuid=str(uuid.uuid4()),
    item_type="text",
    data=["Field note: observed signs of mammal activity near the stream."],
    geometry={"type": "Point", "coordinates": [-1.5, 53.4]},
)

feature = build_feature_record(
    feature_uuid=str(uuid.uuid4()),
    project_system_id=1,
    procedure_id=1,
    start_time=datetime.now(tz=timezone.utc) - timedelta(minutes=5),
    end_time=datetime.now(tz=timezone.utc),
    created_by_method="drawn",
    geometry={"type": "Point", "coordinates": [-1.5, 53.4]},
    observations=[observation],
)

validation = validate_observation_payload([feature], device)
print(validation)

if RUN_UPLOADS and validation["valid"] and PROJECT_ID is not None:
    upload_out = upload_phone_observations(
        hdr=hdr,
        project_id=int(PROJECT_ID),
        feature_payload=[feature],
        device_settings=device,
        validate=True,
    )
    print(upload_out["summary"])
else:
    print("Dry run only: set RUN_UPLOADS=True and PROJECT_ID to call upload_phone_observations(...)")

## Next Steps

- Replace demo values (timestamps, taxonomy, geometry, IDs) with real records.
- Re-run each section in dry-run mode and inspect payloads.
- Enable writes by setting `RUN_UPLOADS = True`.